# Box World Reconstruction (2D Plan View)

This notebook reconstructs each generated world from CSV logs:
- diamond outer walls
- rotated box obstacle
- robot start point
- goal point
- robot trajectory in that trial

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon

# Auto-find latest box world CSV
base_dir = Path.cwd()
outputs_dir = base_dir / 'Models' / 'Outputs'
csv_candidates = sorted(outputs_dir.glob('Output_PPO_Ckpt_Box_World_Test_100*.csv'), key=lambda p: p.stat().st_mtime, reverse=True)

if not csv_candidates:
    raise FileNotFoundError(f'No box world CSV found in: {outputs_dir}')

csv_path = csv_candidates[0]
df = pd.read_csv(csv_path)
print(f'Using CSV: {csv_path}')
print(f'Rows: {len(df)}, Trials: {df["trial_id"].nunique() if "trial_id" in df.columns else "N/A"}')
df.head(2)

In [ ]:
def rotated_box_corners(cx, cy, sx, sy, yaw):
    hx, hy = sx / 2.0, sy / 2.0
    local = np.array([
        [-hx, -hy],
        [ hx, -hy],
        [ hx,  hy],
        [-hx,  hy],
    ])
    c, s = np.cos(yaw), np.sin(yaw)
    rot = np.array([[c, -s], [s, c]])
    return local @ rot.T + np.array([cx, cy])

def draw_trial(ax, trial_df):
    # Use first row in trial as scene metadata
    r0 = trial_df.iloc[0]

    wcx = float(r0.get('world_center_x', 3.0))
    wcy = float(r0.get('world_center_y', 0.0))
    wr = float(r0.get('world_l1_radius', 4.0))

    bcx = float(r0.get('box_center_x', wcx))
    bcy = float(r0.get('box_center_y', wcy))
    bsx = float(r0.get('box_size_x', 1.0))
    bsy = float(r0.get('box_size_y', 1.0))
    byaw = float(r0.get('box_yaw_rad', np.pi / 4.0))

    gx = float(r0['goal_x'])
    gy = float(r0['goal_y'])

    # Diamond outer wall vertices
    diamond = np.array([
        [wcx - wr, wcy],
        [wcx, wcy + wr],
        [wcx + wr, wcy],
        [wcx, wcy - wr],
    ])

    # Rotated box
    box_poly = rotated_box_corners(bcx, bcy, bsx, bsy, byaw)

    # Draw scene
    ax.add_patch(Polygon(diamond, closed=True, fill=False, linewidth=2.0, edgecolor='black', label='Outer wall'))
    ax.add_patch(Polygon(box_poly, closed=True, fill=True, alpha=0.35, facecolor='gray', edgecolor='black', label='Box obstacle'))

    # Trajectory
    ax.plot(trial_df['pos_x'].to_numpy(), trial_df['pos_y'].to_numpy(), linewidth=1.2, color='tab:blue', label='Trajectory')

    # Robot start and goal
    ax.scatter([0.0], [0.0], marker='o', s=55, color='tab:orange', label='Robot start')
    ax.scatter([gx], [gy], marker='*', s=120, color='tab:green', label='Goal')

    trial_id = int(r0.get('trial_id', -1))
    ax.set_title(f'Trial {trial_id} | box=({bsx:.2f}, {bsy:.2f}) yaw={byaw:.2f} rad')
    ax.set_aspect('equal', adjustable='box')
    ax.set_xlim(wcx - wr - 0.8, wcx + wr + 0.8)
    ax.set_ylim(wcy - wr - 0.8, wcy + wr + 0.8)
    ax.grid(True, linestyle='--', alpha=0.35)
    ax.set_xlabel('x (m)')
    ax.set_ylabel('y (m)')

def plot_one_trial(df_all, trial_id):
    tdf = df_all[df_all['trial_id'] == trial_id].copy()
    if tdf.empty:
        raise ValueError(f'trial_id={trial_id} not found')
    fig, ax = plt.subplots(figsize=(7, 7))
    draw_trial(ax, tdf)
    handles, labels = ax.get_legend_handles_labels()
    uniq = dict(zip(labels, handles))
    ax.legend(uniq.values(), uniq.keys(), loc='upper right')
    plt.show()

In [ ]:
# Example: plot one trial
all_trials = sorted(df['trial_id'].dropna().astype(int).unique())
print(f'Available trial ids: {all_trials[:10]} ... total={len(all_trials)}')

trial_to_show = all_trials[0]
plot_one_trial(df, trial_to_show)

In [ ]:
# Plot multiple trials in a grid
def plot_trials_grid(df_all, trial_ids, ncols=3):
    n = len(trial_ids)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 6 * nrows))
    axes = np.array(axes).reshape(-1)

    for i, tid in enumerate(trial_ids):
        tdf = df_all[df_all['trial_id'] == tid]
        if tdf.empty:
            axes[i].set_title(f'Trial {tid} (missing)')
            axes[i].axis('off')
            continue
        draw_trial(axes[i], tdf)

    for j in range(n, len(axes)):
        axes[j].axis('off')

    plt.tight_layout()
    plt.show()

plot_trials_grid(df, all_trials[:9], ncols=3)